In [ ]:
'''
Understanding the Need for Bidirectional Relationships
Imagine you have two tables: Author and Book. An author can write multiple books, and a book is written by one author. This is
a typical one-to-many relationship.
   a) Bidirectional Relationship (using back_populates)
    This allows you to traverse the relationship in both directions easily. You can access an author's books and a book's author
    directly and efficiently.

    b) Unidirectional Relationship
    You might define the relationship so you can easily access an author's books (e.g., author.books). However, with just this,
    accessing the author of a book (e.g., book.author) would require a separate, potentially more complex query.
    
In this example:

Book.author = relationship("Author", back_populates="books") does the same from the Book side, connecting the author 
attribute in Book to the books attribute in Author.
Author.books = relationship("Book", back_populates="author") says that the books attribute in the Author class is related to
the Book class, and the reverse of this relationship is represented by the author attribute in the Book class.

'''

In [ ]:
'''
1.
   a) Author and Book classes represent database tables.
   b) relationship() defines the relationship between these tables.
   c) back_populates="author" in Author.books means the reverse of this relationship (accessing the author from a book) is
    represented by the author attribute in the Book class.
   d) back_populates="books" in Book.author does the same for the other direction. 
    It means the reverse relationship (accessing books from an author) is represented by the books attribute in the Author class.

2.Creating Data

    a) The code creates sample Author and Book objects and adds them to the database session.

3. Bidirectional Access

    a) The code retrieves an author and then iterates through their books using retrieved_author.books. This is the forward 
    direction of the relationship.
    b) It then retrieves a book and accesses its author using retrieved_book.author. This is the reverse direction. Without 
    back_populates, this reverse access would be more complex, likely requiring a separate query.

4. Consistency

    The code demonstrates how back_populates maintains consistency. When a new book is added to an author's books collection 
    using retrieved_author.books.append(new_book), SQLAlchemy automatically sets the new_book.author attribute to the correct 
    author. Without back_populates, you would have to manually set new_book.author = retrieved_author, which is more error-prone.

5) Why back_populates is Important

    a) Avoiding Infinite Recursion
    When dealing with complex object graphs and serialization (e.g., converting objects to JSON), back_populates helps 
    SQLAlchemy avoid infinite recursion issues that can arise from bidirectional relationships.
    b) Reduced Code
    You don't need to write extra code to manage the reverse relationships manually.
    c)Data Integrity
    It ensures that both sides of the relationship are kept in sync, preventing inconsistencies. If you add a book to an 
    author's list of books, back_populates automatically sets the book's author attribute.
    d) Simplified Navigation
    It makes it easy to traverse relationships in both directions, making your code more readable and efficient.'''


In [1]:
from sqlalchemy import Column, Integer, String, ForeignKey
from sqlalchemy.orm import relationship, declarative_base, Session
from sqlalchemy import create_engine

Base = declarative_base()

class Author(Base):
    __tablename__ = 'authors'

    id = Column(Integer, primary_key=True)
    name = Column(String)

    books = relationship("Book", back_populates="author")

class Book(Base):
    __tablename__ = 'books'

    id = Column(Integer, primary_key=True)
    title = Column(String)
    author_id = Column(Integer, ForeignKey('authors.id'))

    author = relationship("Author", back_populates="books")

# Create an in-memory SQLite database for demonstration
engine = create_engine('sqlite:///:memory:')
Base.metadata.create_all(engine)
session = Session(engine)


In [2]:
# Create some data
author1 = Author(name="Stephen King")
book1 = Book(title="The Shining", author=author1)
book2 = Book(title="It", author=author1)

author2 = Author(name="J.K. Rowling")
book3 = Book(title="Harry Potter and the Sorcerer's Stone", author=author2)

session.add_all([author1, book1, book2, author2, book3])
session.commit()

In [3]:
# Example usage demonstrating bidirectional relationships
retrieved_author = session.query(Author).filter_by(name="Stephen King").first()

# Accessing books from the author (forward direction)
print(f"{retrieved_author.name}'s books:")
for book in retrieved_author.books:
    print(f"- {book.title}")

Stephen King's books:
- The Shining
- It


In [4]:
retrieved_book = session.query(Book).filter_by(title="It").first()

# Accessing the author from the book (reverse direction)
print(f"\n{retrieved_book.title}'s author:")
print(f"- {retrieved_book.author.name}")


It's author:
- Stephen King


In [5]:
# Demonstrating how back_populates keeps things consistent
new_book = Book(title="Carrie")
retrieved_author.books.append(new_book)  # Add the book to the author's books collection
session.commit()

print(f"\n{retrieved_author.name}'s books (after adding Carrie):")
for book in retrieved_author.books:
    print(f"- {book.title}")

print(f"\n{new_book.title}'s author:")
print(f"- {new_book.author.name}") 


Stephen King's books (after adding Carrie):
- The Shining
- It
- Carrie

Carrie's author:
- Stephen King
